In [43]:
import route_matrix
import json
import tqdm

In [44]:
coordinates = json.load(open("data/tract-centroids.json"))
pairs = json.load(open("data/origin_pairs.json"))

In [45]:
def deduplicate(l):
    new_l = []
    for e in l:
        if not e in new_l:
            new_l.append(e)
    return new_l


origin_coords = {k: coordinates[k] for k, v in pairs.items()}
destination_coords = {k: [coordinates[vi] for vi in deduplicate(v)] 
 for k,v in pairs.items()}

destination_coords

{'Census Tract 814.02': [[-87.6853728497224, 41.92862970854046],
  [-87.61975201545292, 41.89503904573094],
  [-87.64296872542037, 41.86789230931981],
  [-87.60996759660905, 41.89190829618846],
  [-87.62517035037656, 41.834959971716984],
  [-87.63274641803926, 41.88102686254463]],
 'Census Tract 813': [[-87.64262363419903, 41.87924795024878],
  [-87.60996759660905, 41.89190829618846],
  [-87.62623002479897, 41.89255912578125],
  [-87.62621204437693, 41.89915380624777],
  [-87.6428042439965, 41.88529624776741],
  [-87.62093223737456, 41.88504648048074],
  [-87.637795508798, 41.89325552522301]],
 'Census Tract 208.01': [[-87.6716805484159, 41.87866521261404],
  [-87.70368969259569, 41.986891558552614],
  [-87.90389546005812, 41.98026693652026],
  [-87.68102501626824, 41.906771599037214],
  [-87.62093223737456, 41.88504648048074]],
 'Census Tract 2410': [[-87.61975201545292, 41.89503904573094],
  [-87.70071234755646, 41.86075009643528],
  [-87.61736675901997, 41.87772138115119],
  [-87.67

In [46]:
for i, (o, c) in enumerate(origin_coords.items()):
    if i: continue
    lon, lat = c

    print(lat)

41.89297495363672


In [47]:
def get_num_signals(city, nodes_visited):
    def has_signal(node):
        attrs = city.nodes[node]
        if not 'highway' in attrs:
            return False
        elif attrs['highway'] == "traffic_signals":
            return True
        else:
            return False

    num_signals = sum([has_signal[p] for p in nodes_visited])
    return num_signals

In [48]:
total_pairs = 0

for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]

    total_pairs += len(destinations)

print(f"TOTAL COMBINATIONS: {total_pairs}")

664it [00:00, 679604.16it/s]

TOTAL COMBINATIONS: 3796


In [ ]:
import datetime as dt
from route_matrix import compute_travel_times, load_api_key
import os
import time

if os.path.exists('./data/real_traffic_paths_traffic_aware.json'):
    results = json.load(open('./data/real_traffic_paths_traffic_aware.json', 'r'))
else:
    results = {}
    
for i, (o, c) in tqdm.tqdm(enumerate(origin_coords.items())):
    #print(i, o, c)
    if o in results: continue
    if i == 345: continue

    lon, lat = c

    origin = (lat, lon)
    destinations = [tuple([l[1], l[0]]) for l in destination_coords[o]]
    partial_res = compute_travel_times(origin, destinations,
                     dt.datetime.fromisoformat("2026-11-24T05:00:00-06:00"), load_api_key('key'),
                     routing_preference="TRAFFIC_AWARE")

    #print(origin)
    #print(destinations)

    for i, r in enumerate(partial_res):
        r['destination'] = deduplicate(pairs[o])[i]
        r['coords'] = coordinates[r['destination']]

        partial_res[i] = r

    results[o] = partial_res

    json.dump(results, open('./data/real_traffic_paths_traffic_aware.json', 'w'), indent=2)

210it [00:44,  4.77it/s]


KeyboardInterrupt: 

In [42]:
import plotly.graph_objects as go
import numpy as np

signal_dict = json.load(open('data/num_traffic_signals.json', 'r'))
budget_results = json.load(open('results/paths/budget/commutes_old.json'))
results = json.load(open('./data/real_traffic_paths.json', 'r'))

x = []
y = []

for o, r in results.items():
    for d in r:
        dest = d['destination']

        if not o in budget_results: continue
        if not dest in budget_results[o]: continue
        
        raw_result = budget_results[o][dest]
        realworld_result = d['static_duration_s'] 

        unrestricted_path_index = list(raw_result.keys())[0]
        num_signals = signal_dict[o][dest]

        #if num_signals < 10: continue 

        x += [num_signals]
        if not num_signals:
            y += [0]
        else:
            y += [(realworld_result-raw_result[str(unrestricted_path_index)])/num_signals]


rolling_average = [np.mean(np.array(y)[np.array(x) >= t]) for t in range(1, max(x)+1)] 
x_average = np.nan_to_num([np.mean(np.array(y)[np.array(x) == t]) for t in range(1,max(x)+1)], nan=0)

for i, xe in enumerate(x_average):
    if not xe: x_average[i] = x_average[i-1]

fig = go.Figure()
fig.update(**json.load(open('./analysis/graph_config.json', 'r')))
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="Tract data"))
fig.add_trace(go.Scatter(x=[t for t in range(1, max(x)+1)], y=rolling_average, name="Mean (X >= x)", line_color="#E3170A"))
fig.add_trace(go.Scatter(x=[t for t in range(1, max(x)+1)], y=x_average, name="Mean (X == x)", line_color="#63D471"))

fig.update_yaxes(range=(0, 400), title="Additional time per traffic light")
fig.update_xaxes(range=(-2, 72), title="Number of traffic light encounters")

fig.update_layout(
    font=dict(size=20),                 # global default
    title_font_size=20,
    legend_font_size=16,
    xaxis=dict(title_font_size=20, tickfont_size=20,),
    yaxis=dict(title_font_size=20, tickfont_size=20),
)

fig.show()



mean = np.mean(y)

x = np.array(x)

x = x[x.argsort()[::1]]
y = np.array(y)[x.argsort()[::1]]





print(f"AVERAGE EXTRA TIME PER TRAFFIC LIGHT: {mean}")

AVERAGE EXTRA TIME PER TRAFFIC LIGHT: 35.262595368581515


In [37]:
fig.write_image("./analysis/figures/traffic_signals.png", scale=3)

## Computing signals for origin/destination paths

In [87]:
test = {
    "Census Tract 8412": {
    "Census Tract 207.02": {
      "9": 1299.5094453933395,
      "8": 1302.002729223781,
      "7": 1304.4960130542252,
      "6": 1306.9892968846675,
      "5": 1309.4825807151103,
      "4": 1311.9758645455522,
      "3": 1314.469148375995,
      "2": 1316.962432206439,
      "1": 1319.4557160368809,
      "0": 1321.948999867323
    }
    }
}

print(list(test["Census Tract 8412"]["Census Tract 207.02"].keys())[0])

9
